# Exploratory Data Analysis (EDA)
This notebook performs EDA on the processed datasets to answer critical questions about class imbalance, token length distribution, intervention response diversity, and lexical N-grams.

In [9]:
import pandas as pd
import ast
import re
from collections import Counter
import plotly.express as px

# Load datasets
classifier_df = pd.read_csv('../data/processed/classifier_dataset.csv')
generator_df = pd.read_csv('../data/processed/generator_dataset.csv')
print(f"Loaded {len(classifier_df)} classifier samples and {len(generator_df)} generator samples.")

Loaded 54778 classifier samples and 55896 generator samples.


## 1. Class Imbalance Check

In [10]:
class_counts = classifier_df['label'].value_counts().reset_index()
class_counts.columns = ['label', 'count']
class_counts['label_name'] = class_counts['label'].map({0: 'Non-Toxic (0)', 1: 'Toxic (1)'})

fig = px.pie(class_counts, values='count', names='label_name', title='Class Imbalance (Toxic vs Non-Toxic)')
fig.show()

print(class_counts)

   label  count     label_name
0      0  34921  Non-Toxic (0)
1      1  19857      Toxic (1)


## 2. Token Length Distribution

In [11]:
classifier_df['word_count'] = classifier_df['text'].fillna('').apply(lambda x: len(str(x).split()))

fig2 = px.histogram(classifier_df, x='word_count', color='label', nbins=100, 
                    title='Input Text Word Count Distribution',
                    labels={'word_count': 'Word Count', 'label': 'Toxicity Label'})
fig2.update_layout(barmode='overlay')
fig2.update_traces(opacity=0.75)
fig2.show()

print("Input Text Stats:")
print(classifier_df['word_count'].describe())

Input Text Stats:
count    54778.000000
mean        34.144273
std         54.506595
min          1.000000
25%          9.000000
50%         20.000000
75%         41.000000
max       1518.000000
Name: word_count, dtype: float64


In [12]:
generator_df['response_length'] = generator_df['response'].fillna('').apply(lambda x: len(str(x).split()))

fig3 = px.histogram(generator_df, x='response_length', nbins=50, 
                    title='Human Response Word Count Distribution')
fig3.show()

print("Responses Stats:")
print(generator_df['response_length'].describe())

Responses Stats:
count    55896.000000
mean        15.639581
std          5.264339
min          0.000000
25%         12.000000
50%         16.000000
75%         20.000000
max         33.000000
Name: response_length, dtype: float64


## 3. Intervention Response Analysis

In [13]:
all_responses = generator_df['response'].dropna().tolist()

response_counter = Counter(all_responses)
print(f"Total human responses: {len(all_responses)}")
print(f"Unique human responses: {len(response_counter)}")
print("\nTop 10 most common responses:")
top_10 = response_counter.most_common(10)
for r, c in top_10:
    print(f"{c} times: {r[:100]}...")

Total human responses: 55781
Unique human responses: 29084

Top 10 most common responses:
372 times: use of the n-word is unacceptable in our discourse as it demeans and insults blacks....
360 times: use of the r-word is unacceptable in our discourse as it demeans and insults people with mental disa...
318 times: use of the slur here is unacceptable in our discourse as it demeans and insults gays....
235 times: hey there, sexist comments and views are rude and unwelcome. please respect woman and refrain from t...
226 times: use of the slurs here is unacceptable in our discourse as it demeans and insults and alienates other...
203 times: hey there, derogatory comments are not productive. please refrain from using them and consider other...
191 times: use of the c-word is unacceptable in our discourse as it demeans and insults women....
165 times: mentally challenged individuals tend to be highly intelligent so to use that as a way to describe so...
155 times: you can make the same point

## 4. Lexical and N-Gram Analysis

In [14]:
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

toxic_texts = classifier_df[classifier_df['label'] == 1]['text'].dropna().tolist()

def get_ngrams(text_list, n):
    ngrams = []
    for text in text_list:
        words = str(text).translate(str.maketrans('', '', string.punctuation)).split()
        words = [w for w in words if w.strip() and w.lower() not in ENGLISH_STOP_WORDS]
        for i in range(len(words)-n+1):
            ngrams.append(" ".join(words[i:i+n]))
    return Counter(ngrams)

bigrams = get_ngrams(toxic_texts, 2)
trigrams = get_ngrams(toxic_texts, 3)

top_bigrams_df = pd.DataFrame(bigrams.most_common(15), columns=['Bigram', 'Count'])
fig4 = px.bar(top_bigrams_df, x='Count', y='Bigram', orientation='h', title='Top 15 Toxic Bigrams')
fig4.update_layout(yaxis={'categoryorder':'total ascending'})
fig4.show()

top_trigrams_df = pd.DataFrame(trigrams.most_common(15), columns=['Trigram', 'Count'])
fig5 = px.bar(top_trigrams_df, x='Count', y='Trigram', orientation='h', title='Top 15 Toxic Trigrams')
fig5.update_layout(yaxis={'categoryorder':'total ascending'})
fig5.show()